# 01 · Data Exploration

Explore the synthetic marketplace: catalog, price structure, funnel and position bias. Every cell calls the package rather than reimplementing logic.

> Synthetic data — methodology demo, not CARS24 results. Run `make pipeline-small` first so artifacts exist.

In [ ]:
from driveintent.config import load_config
from driveintent.dashboard.state import raw_table  # cached parquet loaders
import pandas as pd, plotly.express as px
cfg = load_config()
cars, users = raw_table('cars'), raw_table('users')
events, imp = raw_table('events'), raw_table('impressions')
print({t: len(raw_table(t)) for t in ['cars','users','sessions','events']})

## Price vs age and kilometres

In [ ]:
cars = cars.assign(age=2026-cars['manufacturing_year'])
px.box(cars, x='age', y='listed_price', title='Listed price by vehicle age').show()
px.scatter(cars, x='kilometres_driven', y='listed_price', color='body_type',
           opacity=0.5, title='Price vs kilometres').show()

## Conversion funnel (SQL analytics)

In [ ]:
from driveintent.data import load_database as db
db.initialize(cfg)  # idempotent
funnel = db.run_sql_file(cfg, 'funnel_analysis.sql')
funnel

## Position bias: CTR decays with list position

In [ ]:
ctr = imp.groupby('list_position')['clicked'].mean().reset_index()
px.line(ctr, x='list_position', y='clicked', markers=True,
        title='CTR by list position (examination bias)').show()